# Activity 3: DataFrame Cleaning Lab

**Module:** Week 3 Day 1  
**Estimated time:** 60 to 70 minutes  
**Format:** Pair activity  

## Objective

Clean a small taxi review dataset in pandas, then apply the same cleaning decisions in Polars.

By the end, you should be able to:

1. Inspect a small messy CSV.
2. Apply one cleaning decision at a time.
3. Explain why each cleaning rule exists.
4. Compare pandas and Polars results.

This is an AI-Free Zone activity. Use your notes, docs, and partner conversation. Do not use an AI assistant to complete the code.


## Part 0: Start Small and Visible

The dataset is intentionally tiny. We are learning the mechanics first.

Run one cell, inspect the output, and explain what changed before moving on.


In [ ]:
from pathlib import Path

import pandas as pd
import polars as pl

DATA_PATH = Path("data/taxi_trip_review.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Make sure you are running this notebook from the project root "
        "and that data/taxi_trip_review.csv exists."
    )

print("Data file found:", DATA_PATH)


## Part 1: pandas Load and Inspect

Start with pandas because it is the baseline tool many Python data teams already use.


In [ ]:
taxi_pd = pd.read_csv(DATA_PATH)
taxi_pd.head()


In [ ]:
print("shape:", taxi_pd.shape)
print("columns:", list(taxi_pd.columns))


In [ ]:
taxi_pd.dtypes


In [ ]:
taxi_pd.isna().sum()


In [ ]:
print("duplicate trip_id rows:", taxi_pd.duplicated(subset=["trip_id"]).sum())


## Part 2: pandas Clean, One Decision at a Time

Copy first. Then change one thing at a time and inspect the result.


### Cleaning decision: column name standardization

Business rule:
- Column names should be lowercase and should not have leading or trailing spaces.

Why:
Consistent column names make later code easier to read and reduce typing mistakes.


In [ ]:
taxi_pd_clean = taxi_pd.copy()

# TODO:
# Clean the column names by removing leading/trailing spaces and making them lowercase.
# Save the cleaned column names back to taxi_pd_clean.columns.

# Your code here

taxi_pd_clean.columns


### Cleaning decision: text trimming

Business rule:
- Text identifier and category columns should use pandas string dtype.
- Leading and trailing spaces should be removed.

Why:
Extra spaces can create false categories such as `card` and ` card`.


In [ ]:
TEXT_COLUMNS = ["trip_id", "vendor", "pickup_borough", "dropoff_borough", "payment_type"]

# TODO:
# For each column in TEXT_COLUMNS:
# 1. Convert it to a pandas string dtype
# 2. Remove leading/trailing spaces
# 3. Save the cleaned values back to taxi_pd_clean[column]

# Your code here

taxi_pd_clean[TEXT_COLUMNS].head()


### Cleaning decision: datetime parsing

Business rule:
- `pickup_ts` should be parsed as a real datetime.
- Invalid datetime values should become missing values.

Why:
Datetime columns unlock time-based features such as pickup hour. Invalid dates should be visible instead of silently trusted.


In [ ]:
# TODO:
# Convert pickup_ts to datetime.
# Use errors="coerce" so invalid dates become missing values.
# Use utc=True so timestamps have a consistent timezone.

# Your code here

print("invalid pickup_ts after parsing:", taxi_pd_clean["pickup_ts"].isna().sum())
taxi_pd_clean[["trip_id", "pickup_ts"]].head(10)


### Cleaning decision: numeric conversion

Business rule:
- `fare_amount`, `tip_amount`, and `trip_distance` should be numeric.
- Invalid numeric values should become missing values.

Why:
Math operations such as addition, averages, and comparisons only work reliably on numeric columns.


In [ ]:
NUMERIC_COLUMNS = ["fare_amount", "tip_amount", "trip_distance"]

# TODO:
# For each column in NUMERIC_COLUMNS:
# Convert it to numeric values.
# Use errors="coerce" so invalid values become missing values.

# Your code here

taxi_pd_clean[NUMERIC_COLUMNS].dtypes


### Cleaning decision: missing or negative tips

Business rule:
- Missing tip amounts are treated as 0.0.
- Negative tip amounts are not valid for this activity, so replace them with 0.0.

Why:
A missing or negative tip would distort total amount and tip rate calculations.


In [ ]:
print("missing tips before fill:", taxi_pd_clean["tip_amount"].isna().sum())
print("negative tips before fix:", (taxi_pd_clean["tip_amount"] < 0).sum())

# TODO:
# Replace missing tip_amount values with 0.0.
# Replace negative tip_amount values with 0.0.

# Your code here

print("missing tips after fix:", taxi_pd_clean["tip_amount"].isna().sum())
print("negative tips after fix:", (taxi_pd_clean["tip_amount"] < 0).sum())


### Cleaning decision: required fields

Business rule:
- Rows must have the fields needed for this analysis.
- Drop rows missing `trip_id`, boroughs, numeric fare and distance, or parsed pickup time.

Why:
Rows missing required fields cannot be trusted for borough-level trip summaries.


In [ ]:
REQUIRED_COLUMNS = [
    "trip_id",
    "vendor",
    "pickup_borough",
    "dropoff_borough",
    "fare_amount",
    "trip_distance",
    "pickup_ts",
    "payment_type",
]

before = len(taxi_pd_clean)

# TODO:
# Drop rows with missing values in REQUIRED_COLUMNS.
# Save the result back to taxi_pd_clean.

# Your code here

print("after required fields:", len(taxi_pd_clean), "dropped:", before - len(taxi_pd_clean))


### Cleaning decision: positive fare and distance

Business rule:
- `fare_amount` must be greater than 0.
- `trip_distance` must be greater than 0.

Why:
Zero or negative values do not fit this simple trip analysis and can break averages.


In [ ]:
before = len(taxi_pd_clean)

# TODO:
# Keep only rows where fare_amount > 0 and trip_distance > 0.
# Use .loc[...] and add .copy() after filtering.

# Your code here

print("after positive fare and distance:", len(taxi_pd_clean), "dropped:", before - len(taxi_pd_clean))


### Cleaning decision: duplicate trip IDs

Business rule:
- Keep the first row for each `trip_id`.
- Remove later duplicate rows with the same `trip_id`.

Why:
Duplicate trip IDs would double-count trips and inflate revenue.


In [ ]:
before = len(taxi_pd_clean)

# TODO:
# Drop duplicate trip_id rows and keep the first copy.
# Add .copy() after duplicate removal.

# Your code here

print("after duplicate trip_id removal:", len(taxi_pd_clean), "dropped:", before - len(taxi_pd_clean))


### Cleaning decision: derived columns

Business rule:
- Add `pickup_hour` from the parsed timestamp.
- Add `total_amount` as fare plus tip.
- Add `tip_rate` as tip divided by fare.

Why:
Cleaned columns become more useful when we derive simple analysis features from them.


In [ ]:
# TODO:
# Create pickup_hour, total_amount, and tip_rate.

# Your code here

taxi_pd_clean[["trip_id", "pickup_hour", "fare_amount", "tip_amount", "total_amount", "tip_rate"]].head()


Expected counts:

```text
raw rows: 16
clean rows: 11
```


## Part 3: pandas Summary

Now that you trust the cleaned rows, summarize by pickup borough.


In [ ]:
# TODO:
# Group taxi_pd_clean by pickup_borough.
# Create pandas_summary with trip_count, total_revenue, avg_fare, and avg_distance.
# Sort by pickup_borough.

# Your code here

pandas_summary


## Part 4: Polars Load and Inspect

Polars is a different DataFrame library. It often uses expressions such as `pl.col(...)` rather than pandas-style column indexing.


In [ ]:
taxi_pl = pl.read_csv(DATA_PATH)
print("shape:", taxi_pl.shape)
print("columns:", taxi_pl.columns)
taxi_pl.head()


In [ ]:
print("dtypes:", taxi_pl.dtypes)
taxi_pl.null_count()


## Part 5: Polars Clean, Same Decisions

Use the same cleaning policy you used in pandas. The syntax changes, but the decisions should not.


### Cleaning decision: column name standardization

Business rule:
- Column names should be lowercase and should not have leading or trailing spaces.

Why:
The same column naming rule should apply no matter which DataFrame library you use.


In [ ]:
# TODO:
# Rename the Polars columns so they are stripped and lowercase.
# Save the result as taxi_pl_clean.

# Your code here

taxi_pl_clean.columns


### Cleaning decision: text trimming

Business rule:
- Text identifier and category columns should be strings with leading and trailing spaces removed.

Why:
The pandas and Polars results should group the same real categories.


In [ ]:
# TODO:
# Use with_columns and pl.col(...).cast(pl.Utf8).str.strip_chars()
# to clean every column in TEXT_COLUMNS.

# Your code here

taxi_pl_clean.select(TEXT_COLUMNS).head()


### Cleaning decision: datetime and numeric parsing

Business rule:
- Parse `pickup_ts` as datetime.
- Cast fare, tip, and distance columns to numeric values.
- Invalid values should become missing values.

Why:
The same invalid values found in pandas should be handled in Polars too.


In [ ]:
# TODO:
# Use with_columns to parse pickup_ts and cast numeric columns.
# Use strict=False so invalid values become missing values.

# Your code here

taxi_pl_clean.select(["pickup_ts", "fare_amount", "tip_amount", "trip_distance"]).head(10)


### Cleaning decision: missing or negative tips

Business rule:
- Missing tip amounts are treated as 0.0.
- Negative tip amounts are replaced with 0.0.

Why:
The total amount and tip rate rules should match the pandas version.


In [ ]:
# TODO:
# Use pl.when(...).then(...).otherwise(...) to replace missing or negative tips with 0.0.

# Your code here

taxi_pl_clean.select("tip_amount").describe()


### Cleaning decision: required fields

Business rule:
- Drop rows missing the required analysis fields.

Why:
The Polars cleaned table should represent the same valid records as pandas.


In [ ]:
before = taxi_pl_clean.height

# TODO:
# Drop rows with missing values in REQUIRED_COLUMNS.

# Your code here

print("after required fields:", taxi_pl_clean.height, "dropped:", before - taxi_pl_clean.height)


### Cleaning decision: positive fare and distance

Business rule:
- `fare_amount` must be greater than 0.
- `trip_distance` must be greater than 0.

Why:
The same positive-value policy should be applied in both libraries.


In [ ]:
before = taxi_pl_clean.height

# TODO:
# Filter to rows where fare_amount > 0 and trip_distance > 0.

# Your code here

print("after positive fare and distance:", taxi_pl_clean.height, "dropped:", before - taxi_pl_clean.height)


### Cleaning decision: duplicate trip IDs

Business rule:
- Keep the first row for each `trip_id` in the current order.

Why:
Using `maintain_order=True` makes the Polars behavior match the pandas keep-first idea more clearly.


In [ ]:
before = taxi_pl_clean.height

# TODO:
# Remove duplicate trip_id rows.
# Use keep="first" and maintain_order=True.

# Your code here

print("after duplicate trip_id removal:", taxi_pl_clean.height, "dropped:", before - taxi_pl_clean.height)


### Cleaning decision: derived columns

Business rule:
- Add the same `pickup_hour`, `total_amount`, and `tip_rate` columns.

Why:
Matching derived columns helps confirm the pandas and Polars workflows are equivalent.


In [ ]:
# TODO:
# Use with_columns to create pickup_hour, total_amount, and tip_rate.

# Your code here

taxi_pl_clean.select(["trip_id", "pickup_hour", "fare_amount", "tip_amount", "total_amount", "tip_rate"]).head()


## Part 6: Polars Summary and Check


In [ ]:
# TODO:
# Create polars_summary with trip_count, total_revenue, avg_fare, and avg_distance.
# Sort by pickup_borough.

# Your code here

polars_summary


In [ ]:
pandas_check = pandas_summary.sort_values("pickup_borough").reset_index(drop=True)

polars_check = (
    polars_summary
    .sort("pickup_borough")
    .to_pandas()
    .reset_index(drop=True)
)

print("Pandas clean rows:", len(taxi_pd_clean))
print("Polars clean rows:", taxi_pl_clean.height)

assert len(taxi_pd_clean) == taxi_pl_clean.height
assert list(pandas_check["pickup_borough"]) == list(polars_check["pickup_borough"])
assert list(pandas_check["trip_count"]) == list(polars_check["trip_count"])

print("Basic Pandas and Polars checks passed.")


## Part 7: What About Other DataFrame Engines?

pandas and Polars are the two core engines for this activity. Other DataFrame tools exist, but this lab is not a multi-engine benchmark.

| Tool | Useful idea | Why we are not coding it here |
|---|---|---|
| Dask | pandas-like API for larger-than-memory or distributed work | It adds scheduling concepts that distract from today's cleaning practice |
| Modin | speeds up some pandas workloads with minimal code changes | It depends on execution backends that need separate setup |
| FireDucks | accelerates some pandas workloads | It is optional and environment-dependent |

The main lesson: learn the cleaning decisions first. Engines can change later.


## Reflection

Answer these in your Day 1 project README:

1. Which pandas cleaning step changed the row count the most?
2. Which Polars expression felt most different from pandas?
3. Why is it useful to write the cleaning policy in plain English before coding it?
